# LeRobot Dataset Validation & Stationarity Analysis

This notebook provides a comprehensive statistical pipeline to validate imitation learning datasets. It is divided into two phases:
* **Phase 1 (Within-Episode):** Checks action and observation increments (deltas) for difference-stationarity to ensure hardware signal smoothness.
* **Phase 2 (Across-Episode):** Groups episodes chronologically to test for expert strategy drift, fatigue, and environment initialization bias using Kruskal-Wallis testing.

In [ ]:
# Run this cell once to install the required dependencies
!pip install -q lerobot statsmodels pandas numpy scipy tqdm matplotlib seaborn

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import kruskal
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

from lerobot.datasets.lerobot_dataset import LeRobotDataset
from statsmodels.tsa.stattools import adfuller, kpss

# Ensure plots render inline
%matplotlib inline
sns.set_theme(style="whitegrid")

# =============================================================================
# CONFIGURATION
# =============================================================================
REPO_ID = "unitreerobotics/G1_Brainco_PickDrink_Dataset" # Adjust if using a local fork
ROOT = Path("./my_local_data_dir")
EPISODES = None          # None = all episodes
ALPHA = 0.05
MIN_LEN = 20
NUM_BUCKETS = 3          # Chronological groups for drift testing
OUT_DIR = Path("./dataset_validation_out")

# --- STATIONARITY TUNING ---
DIFF_ORDER = 1           # 1 = velocity (deltas)
ADF_AUTOLAG = "AIC"      
MAX_LAG = None           
KPSS_NLAGS = "auto"      
# =============================================================================

In [ ]:
def classify(adf_p: float, kpss_p: float, alpha: float = ALPHA) -> str:
    adf_stationary = adf_p < alpha
    kpss_stationary = kpss_p >= alpha
    if adf_stationary and kpss_stationary: return "stationary"
    if (not adf_stationary) and (not kpss_stationary): return "non_stationary"
    if adf_stationary and (not kpss_stationary): return "difference_stationary"
    return "trend_stationary"

def safe_adf(x: np.ndarray) -> float:
    if np.allclose(x, x[0]): return np.nan
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            return float(adfuller(x, maxlag=MAX_LAG, autolag=ADF_AUTOLAG)[1])
    except Exception: return np.nan

def safe_kpss(x: np.ndarray) -> float:
    if np.allclose(x, x[0]): return np.nan
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            return float(kpss(x, regression="c", nlags=KPSS_NLAGS)[1])
    except Exception: return np.nan

def get_episode_matrix(dataset: LeRobotDataset, ep_idx: int, key: str) -> np.ndarray | None:
    if key not in dataset.hf_dataset.features: return None
    from_idx = int(dataset.meta.episodes["dataset_from_index"][ep_idx])
    to_idx = int(dataset.meta.episodes["dataset_to_index"][ep_idx])
    data = dataset.hf_dataset.select(range(from_idx, to_idx))[key]
    return np.asarray(data, dtype=np.float64)

def test_across_episode_drift(metrics_df: pd.DataFrame, num_buckets: int) -> pd.DataFrame:
    metrics_df = metrics_df.sort_values("episode").reset_index(drop=True)
    metrics_df["bucket"] = pd.qcut(metrics_df.index, q=num_buckets, labels=False)
    drift_results = []
    cols_to_test = [c for c in metrics_df.columns if c not in ["episode", "bucket"]]
    
    for col in cols_to_test:
        groups = [group[col].values for _, group in metrics_df.groupby("bucket")]
        if len(groups) > 1 and all(len(g) > 0 for g in groups):
            try:
                stat, p_val = kruskal(*groups)
                drift_results.append({"metric": col, "p_value": p_val, "drift_detected": p_val < ALPHA})
            except ValueError:
                pass
    return pd.DataFrame(drift_results)

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Loading dataset repo_id={REPO_ID!r} from root={ROOT}")
dataset = LeRobotDataset(repo_id=REPO_ID, root=ROOT)
episodes = EPISODES if EPISODES is not None else list(range(dataset.num_episodes))

within_ep_rows = []
across_ep_metrics = []

for ep in tqdm(episodes, desc="Validating Episodes"):
    A_action = get_episode_matrix(dataset, ep, "action")
    A_obs = get_episode_matrix(dataset, ep, "observation.state")
    
    if A_action is None or A_action.ndim != 2 or A_action.shape[0] < MIN_LEN: continue

    ep_metrics = {"episode": ep, "trajectory_length": A_action.shape[0]}
    if A_obs is not None and A_obs.shape[0] > 0:
        for d in range(A_obs[0, :].shape[0]):
            ep_metrics[f"init_obs_dim_{d}"] = A_obs[0, d]

    data_streams = {"action": A_action}
    if A_obs is not None: data_streams["observation.state"] = A_obs

    for stream_name, raw_matrix in data_streams.items():
        data_to_test = np.diff(raw_matrix, n=DIFF_ORDER, axis=0) if DIFF_ORDER > 0 else raw_matrix
        for d in range(data_to_test.shape[1]):
            x_series = data_to_test[:, d]
            ep_metrics[f"var_{stream_name}_dim_{d}"] = np.sum(np.abs(x_series))
            
            adf_p = safe_adf(x_series)
            kpss_p = safe_kpss(x_series)
            verdict = "constant" if (np.isnan(adf_p) and np.isnan(kpss_p)) else classify(
                adf_p if not np.isnan(adf_p) else 1.0, kpss_p if not np.isnan(kpss_p) else 0.0)
            
            within_ep_rows.append({"episode": ep, "stream": stream_name, "dim": d, "adf_p": adf_p, "kpss_p": kpss_p, "verdict": verdict})
            
    across_ep_metrics.append(ep_metrics)

# Save to memory and disk
df_within = pd.DataFrame(within_ep_rows)
df_within.to_csv(OUT_DIR / f"within_episode_d{DIFF_ORDER}.csv", index=False)

df_metrics = pd.DataFrame(across_ep_metrics)
df_drift = test_across_episode_drift(df_metrics, num_buckets=NUM_BUCKETS)
df_drift.to_csv(OUT_DIR / "across_episode_drift.csv", index=False)

print("✅ Statistical testing complete. Data saved.")

In [ ]:
# Create the figure
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 12), gridspec_kw={'height_ratios': [2, 1]})
plt.subplots_adjust(hspace=0.4)

# --- Plot Phase 1: Heatmap ---
verdict_map = {"stationary": 0, "difference_stationary": 1, "trend_stationary": 2, "non_stationary": 3, "constant": 4}
df_actions = df_within[df_within["stream"] == "action"].copy()
df_actions["verdict_num"] = df_actions["verdict"].map(verdict_map)
pivot_df = df_actions.pivot(index="dim", columns="episode", values="verdict_num")

cmap = sns.color_palette(["#2ecc71", "#f1c40f", "#e67e22", "#e74c3c", "#95a5a6"])
sns.heatmap(pivot_df, cmap=cmap, ax=ax1, cbar=False, vmin=0, vmax=4)
ax1.set_title("Phase 1: Action Smoothness (Hardware Health)", pad=15, fontweight="bold")
ax1.set_xlabel("Episode Number")
ax1.set_ylabel("Action Dimension")

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#2ecc71", label="Stationary (Smooth)"),
    Patch(facecolor="#f1c40f", label="Diff-Stationary"),
    Patch(facecolor="#e74c3c", label="Non-Stationary (Jitter/Noise)"),
    Patch(facecolor="#95a5a6", label="Constant (No Movement)")
]
ax1.legend(handles=legend_elements, loc="upper right", bbox_to_anchor=(1.25, 1))

# --- Plot Phase 2: Drift ---
if not df_drift.empty:
    df_drift_sorted = df_drift.sort_values("p_value", ascending=True)
    colors = ["#e74c3c" if p < ALPHA else "#3498db" for p in df_drift_sorted["p_value"]]
    sns.barplot(data=df_drift_sorted, x="p_value", y="metric", ax=ax2, palette=colors)
    ax2.axvline(x=ALPHA, color="red", linestyle="--", linewidth=2, label=f"Drift Threshold (α={ALPHA})")
    ax2.set_title("Phase 2: Expert & Environment Drift", pad=15, fontweight="bold")
    ax2.set_xlabel("Kruskal-Wallis p-value (Lower = More Drift)")
    ax2.set_ylabel("Evaluated Metric")
    ax2.legend()
else:
    ax2.text(0.5, 0.5, "Not enough variance in Phase 2 data.", ha='center', va='center')

plt.suptitle("Dataset Validation Dashboard", fontsize=16, fontweight="bold", y=0.95)
plt.show()